# RS-Mamba Training Baseline (LEVIR-CD) — Phase 3

Stands up the PyTorch RS-Mamba change-detection baseline on **LEVIR-CD**, running top-to-bottom on a **Kaggle Tesla T4**. This baseline doubles as the **throughput reference** that TensorRT must beat in Phase 5.

**Decisions for this run** (see `CLAUDE.md`):
- **Crops persist in ephemeral `/kaggle/working`** — re-cropped each session, no Kaggle output dataset.
- **Short smoke training (~8 epochs, capped batches/epoch)** — proves the pipeline trains end-to-end; accuracy is a sanity check, not a convergence run.
- **Throughput-first** — the FP32 images/sec benchmark (final section) is the headline deliverable.

**Pipeline:** env + kernel → import vendored code → download LEVIR-CD → crop 1024→256 on disk → loaders → RSM-CD tiny → smoke train → quick IoU/F1 → **throughput benchmark**.

## 1. Environment

The model needs a CUDA selective-scan kernel that isn't bundled — it's compiled from VMamba. The vendored RS-Mamba code lives in **this repo** at `model/vendor/rs_mamba_cd/`, so we clone the repo to get it.

> **Kernel ordering gotcha:** `rs_mamba_cd` binds the kernel *at import time*. We build the kernel **before** importing the model (handled across the next few cells).

In [16]:
import os

WORK = "/kaggle/working"
os.chdir(WORK)

REPO_URL = "https://github.com/fergalriordan/Mamba-RS-Engine.git"
VMAMBA_URL = "https://github.com/MzeroMiko/VMamba.git"

# This repo carries the vendored RS-Mamba code (model/vendor/rs_mamba_cd).
if not os.path.isdir(f"{WORK}/Mamba-RS-Engine"):
    !git clone --depth 1 {REPO_URL}
# VMamba provides the selective-scan CUDA kernel.
if not os.path.isdir(f"{WORK}/VMamba"):
    !git clone --depth 1 {VMAMBA_URL}

VENDOR = f"{WORK}/Mamba-RS-Engine/model/vendor/rs_mamba_cd"
assert os.path.isdir(VENDOR), f"vendored code not found at {VENDOR}"
print("vendor OK:", VENDOR)

Cloning into 'Mamba-RS-Engine'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 29 (delta 0), reused 23 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 2.31 MiB | 15.57 MiB/s, done.
Cloning into 'VMamba'...
remote: Enumerating objects: 1989, done.
remote: Counting objects: 100% (1989/1989), done.
remote: Compressing objects: 100% (1131/1131), done.
remote: Total 1989 (delta 957), reused 1795 (delta 847), pack-reused 0 (from 0)
Receiving objects: 100% (1989/1989), 12.13 MiB | 36.63 MiB/s, done.
Resolving deltas: 100% (957/957), done.
vendor OK: /kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd


In [17]:
# Install deps individually WITHOUT the numpy==1.22.4 / Pillow pins that fail to build on Py3.12.
# torch / torchvision / numpy / PIL / opencv / kagglehub come preinstalled on Kaggle.
!pip install -q "torchmetrics==0.9.3" einops timm fvcore albumentations kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.6/419.6 kB 9.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 wh

In [18]:
# Build the selective-scan CUDA kernel -> installs `selective_scan_cuda_oflex`.
# MUST run BEFORE importing rs_mamba_cd. setup.py compiles MODES=["oflex"].
%cd /kaggle/working/VMamba/kernels/selective_scan
!pip install -q .
%cd /kaggle/working

/kaggle/working/VMamba/kernels/selective_scan
  Preparing metadata (setup.py) ... done
/kaggle/working


## 2. Make vendored code importable

Add the vendor package and its `utils/` to `sys.path`, verify the freshly built kernel imports, then import the model. On a T4 (compute 7.5) a BFloat16 warning is expected — the `oflex` kernel still works.

In [24]:
import sys

VENDOR = "/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd"

# Only the vendor ROOT goes on sys.path so `utils` resolves to the package dir.
# Adding `{VENDOR}/utils` would let `utils/utils.py` be imported as a top-level
# module named `utils`, shadowing the package -> "'utils' is not a package".
bad = f"{VENDOR}/utils"
if bad in sys.path:
    sys.path.remove(bad)
if VENDOR not in sys.path:
    sys.path.insert(0, VENDOR)

# Clear any stale / half-imported modules from earlier attempts so they re-resolve.
for m in [k for k in list(sys.modules) if k == "utils" or k.startswith("utils.")]:
    del sys.modules[m]

# Import torch FIRST so its shared libs (libc10.so etc.) are loaded into the process.
# The compiled kernel links against them; otherwise import fails with
# 'libc10.so: cannot open shared object file'.
import torch

# Then bind the kernel (must happen before importing rs_mamba_cd).
import selective_scan_cuda_oflex
print("kernel OK:", selective_scan_cuda_oflex.__file__)

# Import-ordering safety: if the model was imported before the kernel build, drop & reimport.
if "rs_mamba_cd" in sys.modules:
    del sys.modules["rs_mamba_cd"]

from rs_mamba_cd import RSM_CD
from utils.data_loading import BasicDataset
from utils.losses import FCCDN_loss_without_seg

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

kernel OK: /usr/local/lib/python3.12/dist-packages/selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so
torch 2.10.0+cu128 | cuda True | Tesla T4


/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:130: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:158: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:176: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/kaggle/working/Mamba-RS-Engine/model/vendor/rs_mamba_cd/rs_mamba_cd.py:184: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/kaggle/working/Mamb

## 3. Download LEVIR-CD

Fully code-based via `kagglehub` (needs Internet enabled). Resolves to `.../LEVIR CD` (note the space), with `train/val/test`, each holding `A/` (before), `B/` (after), `label/`.

In [25]:
import kagglehub, glob

base = kagglehub.dataset_download("mdrifaturrahman33/levir-cd")
print("downloaded to:", base)

# Locate the split root robustly (handles the 'LEVIR CD' subdir with a space).
LEVIR_ROOT = None
for c in [base] + glob.glob(os.path.join(base, "*")):
    if all(os.path.isdir(os.path.join(c, s)) for s in ("train", "val", "test")):
        LEVIR_ROOT = c
        break
assert LEVIR_ROOT, f"could not find train/val/test under {base}"
print("LEVIR_ROOT:", LEVIR_ROOT)
for s in ("train", "val", "test"):
    print(f"  {s}: {len(os.listdir(os.path.join(LEVIR_ROOT, s, 'A')))} pairs")

downloaded to: /kaggle/input/datasets/mdrifaturrahman33/levir-cd
LEVIR_ROOT: /kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD
  train: 445 pairs
  val: 64 pairs
  test: 128 pairs


## 4. Crop 1024 → 256 on disk

`BasicDataset` does **no** resizing — it only normalizes + ToTensors — but the tiny config is fixed at `image_size=256`. So 256px tiles must exist on disk first.

Custom crop loop (clearer than the vendored `crop_img`, and avoids its hard-coded `t1/t2/label` names + size-divisibility assertion). We map **A→t1, B→t2, label→label** with matching filename stems. Non-overlapping 4×4 grid → **16 tiles/image** (7120 train / 1024 val / 2048 test).

In [26]:
from PIL import Image

CROP_ROOT = "/kaggle/working/levir_256"
TILE = 256
SRC_TO_DST = {"A": "t1", "B": "t2", "label": "label"}  # vendored loader expects t1/t2/label

def crop_split(split, overlap=0):
    """Crop a split's 1024x1024 tiles into 256x256 tiles on disk.

    Args:
        split: one of 'train', 'val', 'test'.
        overlap: pixel overlap between adjacent tiles (0 = non-overlapping grid).

    Returns:
        Number of 256px tiles written for the split.
    """
    stride = TILE - overlap
    src_split = os.path.join(LEVIR_ROOT, split)
    for dst in SRC_TO_DST.values():
        os.makedirs(os.path.join(CROP_ROOT, split, dst), exist_ok=True)
    files = sorted(os.listdir(os.path.join(src_split, "A")))
    n_tiles, grid = 0, (0, 0)
    for fname in files:
        stem, ext = os.path.splitext(fname)
        imgs = {dst: Image.open(os.path.join(src_split, src, fname))
                for src, dst in SRC_TO_DST.items()}
        w, h = imgs["t1"].size
        xs = list(range(0, w - TILE + 1, stride))
        ys = list(range(0, h - TILE + 1, stride))
        grid = (len(xs), len(ys))
        for yi, y in enumerate(ys):
            for xi, x in enumerate(xs):
                box = (x, y, x + TILE, y + TILE)
                for dst, im in imgs.items():
                    im.crop(box).save(
                        os.path.join(CROP_ROOT, split, dst, f"{stem}_{yi}_{xi}{ext}"))
                n_tiles += 1
    print(f"{split}: {len(files)} imgs -> {n_tiles} tiles ({grid[0]}x{grid[1]} grid)")
    return n_tiles

if not os.path.isdir(CROP_ROOT):
    for sp in ("train", "val", "test"):
        crop_split(sp, overlap=0)
else:
    print("crops already present at", CROP_ROOT, "- skipping")

train: 445 imgs -> 7120 tiles (4x4 grid)
val: 64 imgs -> 1024 tiles (4x4 grid)
test: 128 imgs -> 2048 tiles (4x4 grid)


## 5. DataLoaders

`BasicDataset` reads parallel `t1/t2/label` dirs with matching stems, binarizes labels (`label[label!=0]=1`), Albumentations-normalizes, and (train only) applies random flip/transpose + t1↔t2 swap. Fewer workers than the vendored default (Kaggle has ~2–4 CPUs).

In [29]:
from torch.utils.data import DataLoader
import albumentations as A

# Compat shim: albumentations >=2.0 removed `A.Flip` (random horizontal/vertical
# flip), which the vendored BasicDataset builds in __init__. Recreate an equivalent
# only if it's missing, so the vendored loader works unchanged on new albumentations.
if not hasattr(A, "Flip"):
    def _Flip(p=0.5, **kwargs):
        return A.OneOf([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)], p=p, **kwargs)
    A.Flip = _Flip

BATCH_SIZE = 16  # comfortably fits T4 at 256px; reused for training

def make_loader(split, train, batch_size, num_workers=2):
    root = f"{CROP_ROOT}/{split}"
    ds = BasicDataset(t1_images_dir=f"{root}/t1/", t2_images_dir=f"{root}/t2/",
                      labels_dir=f"{root}/label/", train=train)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=train, drop_last=train,
                        num_workers=num_workers, pin_memory=True)
    return ds, loader

train_ds, train_loader = make_loader("train", True, BATCH_SIZE)
val_ds, val_loader = make_loader("val", False, BATCH_SIZE)
print("train tiles:", len(train_ds), "| val tiles:", len(val_ds))

train tiles: 7120 | val tiles: 1024


## 6. Model — RSM-CD tiny

`forward_type="v3"` routes through the compiled `selective_scan_cuda_oflex` kernel. Inputs are two `(B,3,256,256)` bitemporal tensors → output `(B,1,256,256)` per-pixel change logits.

In [30]:
device = "cuda"
net = RSM_CD(
    dims=96, depths=[2, 2, 9, 2],
    ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank="auto",
    mlp_ratio=4.0, drop_path_rate=0.2,
    forward_type="v3",
).to(device)
print(f"RSM-CD tiny: {sum(p.numel() for p in net.parameters())/1e6:.2f}M params")

RSM-CD tiny: 51.95M params


## 7. Loss & metrics

**Loss:** vendored `FCCDN_loss_without_seg` = soft-Dice + BCE-with-logits, returning `(total, dice, bce)`. Appropriate given the heavy ~4–5% positive-pixel imbalance.

**Metrics:** we report **change-class** F1 / IoU (not pixel accuracy, which is ~95% trivially). Computed manually from accumulated TP/FP/FN for clarity and to sidestep torchmetrics 0.9.3 binary-API quirks (torchmetrics is still installed if you prefer it).

In [33]:
def new_counts():
    return {"tp": 0, "fp": 0, "fn": 0, "tn": 0}

@torch.no_grad()
def update_counts(counts, logits, labels):
    """Accumulate change-class TP/FP/FN/TN from raw logits and {0,1} labels."""
    preds = (torch.sigmoid(logits.squeeze(1)) > 0.5).long()
    tgt = labels.long()
    counts["tp"] += int(((preds == 1) & (tgt == 1)).sum())
    counts["fp"] += int(((preds == 1) & (tgt == 0)).sum())
    counts["fn"] += int(((preds == 0) & (tgt == 1)).sum())
    counts["tn"] += int(((preds == 0) & (tgt == 0)).sum())

def metrics_from_counts(c):
    """Change-class precision / recall / F1 / IoU from accumulated counts."""
    tp, fp, fn = c["tp"], c["fp"], c["fn"]
    prec = tp / (tp + fp + 1e-9)
    rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    iou = tp / (tp + fp + fn + 1e-9)
    return {"precision": prec, "recall": rec, "f1": f1, "iou": iou}

## 8. Smoke training

Short run to confirm the pipeline trains end-to-end (loss falls, change-class IoU becomes non-trivial). Hyperparams follow the vendored defaults: AdamW lr `1e-3`, wd `1e-3`, AMP on, grad-clip `20`. Batches/epoch are capped to keep the smoke run fast — raise `EPOCHS` / `MAX_TRAIN_BATCHES` (or set the cap to `None`) for a real run.

In [35]:
import time
from torch import optim
from tqdm.auto import tqdm

EPOCHS = 5
MAX_TRAIN_BATCHES = 150   # cap per epoch for a fast smoke run; set None to use the full epoch
GRAD_CLIP = 20.0

optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-3)
scaler = torch.amp.GradScaler("cuda")

steps_per_epoch = (len(train_loader) if MAX_TRAIN_BATCHES is None
                   else min(len(train_loader), MAX_TRAIN_BATCHES))

for epoch in range(EPOCHS):
    net.train()
    running, steps, t0 = 0.0, 0, time.time()
    pbar = tqdm(train_loader, total=steps_per_epoch, desc=f"epoch {epoch+1}/{EPOCHS}", leave=True)
    for i, (t1, t2, label, _) in enumerate(pbar):
        if MAX_TRAIN_BATCHES is not None and i >= MAX_TRAIN_BATCHES:
            break
        t1 = t1.to(device, non_blocking=True)
        t2 = t2.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            logits = net(t1, t2)
            loss, dice, bce = FCCDN_loss_without_seg(logits, label)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
        steps += 1
        pbar.set_postfix(loss=f"{running/steps:.4f}")
    pbar.close()
    print(f"epoch {epoch+1}/{EPOCHS}  avg_loss={running/max(steps,1):.4f}  ({time.time()-t0:.1f}s, {steps} steps)")

epoch 1/5:   0%|          | 0/150 [00:00<?, ?it/s]

epoch 1/5  avg_loss=0.9203  (328.1s, 150 steps)


epoch 2/5:   0%|          | 0/150 [00:00<?, ?it/s]

epoch 2/5  avg_loss=0.7897  (326.2s, 150 steps)


epoch 3/5:   0%|          | 0/150 [00:00<?, ?it/s]

epoch 3/5  avg_loss=0.6909  (327.2s, 150 steps)


epoch 4/5:   0%|          | 0/150 [00:00<?, ?it/s]

epoch 4/5  avg_loss=0.6525  (329.2s, 150 steps)


epoch 5/5:   0%|          | 0/150 [00:00<?, ?it/s]

epoch 5/5  avg_loss=0.5822  (329.0s, 150 steps)


## 9. Quick validation — change-class IoU / F1

Sanity check that the model learned something beyond the trivial all-background prediction.

In [36]:
net.eval()
counts = new_counts()
with torch.no_grad():
    for t1, t2, label, _ in val_loader:
        t1, t2 = t1.to(device), t2.to(device)
        with torch.amp.autocast("cuda"):
            logits = net(t1, t2)
        update_counts(counts, logits.float(), label.to(device))

m = metrics_from_counts(counts)
print("VAL change-class:", {k: round(v, 4) for k, v in m.items()})

VAL change-class: {'precision': 0.6912, 'recall': 0.8078, 'f1': 0.745, 'iou': 0.5936}


## 10. Throughput baseline — the headline number

The PyTorch reference TensorRT must beat in Phase 5. Measured carefully: `eval()` + `no_grad()`, fixed batch/resolution, discarded **warmup**, `cuda.synchronize()` around timing, averaged over many iterations.

**Record the FP32 img/s** as *the* baseline (an honest fixed-precision reference). FP16 is reported alongside since TensorRT will use it.

In [37]:
@torch.no_grad()
def benchmark(net, batch_size, res=256, fp16=False, warmup=10, iters=50):
    """Return (images_per_sec, ms_per_batch) for a fixed batch/resolution."""
    net.eval()
    x1 = torch.randn(batch_size, 3, res, res, device=device)
    x2 = torch.randn(batch_size, 3, res, res, device=device)
    for _ in range(warmup):
        with torch.amp.autocast("cuda", enabled=fp16):
            net(x1, x2)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters):
        with torch.amp.autocast("cuda", enabled=fp16):
            net(x1, x2)
    torch.cuda.synchronize()
    dt = time.time() - t0
    return batch_size * iters / dt, dt / iters * 1000

print(f"GPU: {torch.cuda.get_device_name(0)}  |  res=256")
print(f"{'precision':>9} {'batch':>6} {'img/s':>10} {'ms/batch':>10}")
for bs in (1, 8, 16):
    for fp16 in (False, True):
        ips, mspb = benchmark(net, bs, fp16=fp16)
        print(f"{'fp16' if fp16 else 'fp32':>9} {bs:>6} {ips:>10.1f} {mspb:>10.2f}")

GPU: Tesla T4  |  res=256
precision  batch      img/s   ms/batch
     fp32      1       16.0      62.63
     fp16      1       14.6      68.65
     fp32      8       14.1     568.29
     fp16      8       23.7     337.33
     fp32     16       14.2    1124.40
     fp16     16       23.1     691.17


---
### Baseline recorded (Tesla T4, res 256, forward-only)

**PyTorch throughput reference = 23.7 img/s** (FP16, batch 8) — the best achievable across the sweep.
FP32 best = 16.0 img/s (batch 1). Phase 5 (TensorRT, FP16) must beat the PyTorch best by >2× (~47 img/s).

**Observation:** FP32 throughput is flat (~14–16 img/s) regardless of batch size, while FP16 scales to
~24 img/s by batch 8 then plateaus. The largely-sequential selective-scan kernel is the suspected
bottleneck — and is itself part of the motivation for the TensorRT engine.